# RSNA Knee Mastery: Interactive Zero-GPU Walkthrough

This notebook provides an interactive walkthrough of the four core mathematical and algorithmic ideas behind the **RSNA Knee Abnormality Detection** solution.

**Key feature**: No GPU, no DICOM files, and no external datasets are required. Everything runs in standard NumPy in seconds.

### Table of Contents
1. **Physical Millimetre Cropping**: Invariance to scanner pixel spacing (140 mm FOV)
2. **Fixed Slots & Sampling**: 96 slices across 5 slots with 2%–98% stack sampling
3. **Class-Specific Attention**: Independent multi-instance pooling for 12 distinct findings
4. **Percentile-Rank Blending**: Ensembling models by rank order with 0.5 failure quarantine

In [ ]:
# Setup & verify installation
import sys
from pathlib import Path

# Ensure src/ is importable even when running directly without pip install -e .
repo_root = Path.cwd().resolve() if (Path.cwd() / "src").exists() else Path.cwd().resolve().parent
if (repo_root / "src").exists() and str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

import numpy as np

from rsna_knee import FINDINGS, total_slices

print("rsna_knee loaded successfully!")
print(f"Total target findings: {len(FINDINGS)}")
print(f"Total fixed slices across slots: {total_slices()}")

## 1. Physical Millimetre Cropping (`CROP_MM = 140.0`)

In medical imaging, hospital scanners acquire images with differing in-plane resolution (e.g. 0.3 mm vs 0.5 mm per pixel).
If we simply resized raw DICOM images to 224x224, tiny ligament tears would be obliterated or stretched unpredictably.

Instead, we crop a fixed physical box of **140 mm**, using the formula:
$$\text{pixels} = \text{round}\left(\frac{140.0}{\text{PixelSpacing}}\right)$$

In [ ]:
from rsna_knee import center_crop_box, crop_pixels

spacings = [0.25, 0.30, 0.40, 0.50, 0.60]
print("Scanner In-Plane Spacing -> Required Pixel Crop (for 140 mm):")
for s in spacings:
    px = crop_pixels(s)
    physical_coverage = px * s
    print(f"  Spacing: {s:.2f} mm/px -> Crop: {px:3d} pixels (covers {physical_coverage:.1f} mm)")

# Compute bounding box inside a standard 512x512 scan:
y0, x0, size = center_crop_box(512, 512, pixel_spacing_mm=0.5)
print(f"\n512x512 matrix at 0.5 mm/px: Centered crop box is ({y0}, {x0}) with size {size}x{size}")

## 2. Fixed Slots & 2%–98% Stack Coverage

A complete study contains multiple series with varying slice counts. The pipeline normalizes every study into **5 fixed slots totaling 96 slices**:
- Sagittal Fluid-Sensitive (26 slices)
- Sagittal Non-Fluid (22 slices)
- Coronal Fluid-Sensitive (18 slices)
- Coronal Non-Fluid (12 slices)
- Axial (18 slices)

Crucially, slices are sampled across **2% to 98%** of each series volume to ensure collateral ligaments and outer meniscal horns are not discarded.

In [ ]:
from rsna_knee import SLOTS, pick_indices

for plane, pref, n in SLOTS:
    pref_str = "Fluid-sensitive" if pref == 1 else ("Anatomical" if pref == 0 else "Standard")
    print(f"  Slot {plane:8s} | {pref_str:15s} | Slices: {n:2d}")

# Demonstrate sampling 10 slices from a 100-slice stack:
sampled = pick_indices(n_slices=100, k=10)
print(f"\nSampled indices from 100-slice series:\n  {sampled}")
print("Notice index 2 (outer margin) and index 97 (far outer margin) are both included!")

## 3. Class-Specific Attention Pooling

Different pathologies appear on completely different slices. Rather than using one global attention pooling vector across all findings, each finding has its own attention projection column.

$$\text{PooledFeature}_c = \sum_{k=1}^K \text{Attention}_{k, c} \cdot \text{Feature}_k$$

In [ ]:
from rsna_knee import class_attention

rng = np.random.default_rng(42)
K_windows = 4  # 4 2.5D slice windows
F_features = 6  # 6-dim representation per window
demo_findings = ["ACL", "Medial Meniscus", "Effusion"]
N_findings = len(demo_findings)

# Feature representation from backbone:
features = rng.normal(size=(K_windows, F_features))
# Attention projection weights (one column per finding):
att_weights = rng.normal(size=(F_features, N_findings))

attention_matrix = class_attention(features, att_weights)
print("Attention Matrix [Windows x Findings] (each column sums to 1.0):")
print("Window | " + " | ".join(f"{f:15s}" for f in demo_findings))
print("-" * 60)
for i in range(K_windows):
    row_vals = " | ".join(f"{attention_matrix[i, j]:15.3f}" for j in range(N_findings))
    print(f"  W{i}   | {row_vals}")

print("\nNotice: Window 2 might dominate Effusion, while Window 0 dominates ACL!")

## 4. Percentile-Rank Blending & Failure Quarantine

Macro-AUC evaluates ranking accuracy, not calibration. Models are combined via **percentile ranks**:
$$\text{PercentileRank}(x_i) = \frac{\text{rank}(x_i)}{N}$$

### Sentinel Quarantine
If a corrupted study causes Model A to fail, Model A fills its prediction with `0.5`.
In our fusion logic, failed rows are excluded from ranking computation via `pct_rank_on`, and the healthy model's ranking automatically takes over!

In [ ]:
from rsna_knee import fuse_finding, pct_rank

# 5 patients, scores for one finding
model_a = np.array([0.90, 0.85, 0.30, 0.20, 0.10])  # Confident
model_b = np.array([0.55, 0.60, 0.52, 0.51, 0.40])  # Conservative, same ranking
model_c = np.array([0.70, 0.88, 0.25, 0.30, 0.15])  # 3rd model

print(f"Model A Percentile Ranks: {pct_rank(model_a)}")
print(f"Model B Percentile Ranks: {pct_rank(model_b)}")

# Simulate Patient 2 failing in Model A:
f_a = np.array([False, False, True, False, False])  # Patient 2 failed
f_b = np.zeros(5, dtype=bool)
f_c = np.zeros(5, dtype=bool)

fused = fuse_finding(
    raptor384=model_a,
    raptor224=model_b,
    ours=model_c,
    fail_384=f_a,
    fail_224=f_b,
    fail_ours=f_c,
    w_raptor=0.5,
    a224=0.35,
)
print("\nFused Final Score (with Patient #2 safely handled without crashing):")
for idx, val in enumerate(fused):
    tag = " [Model A Failed - Takeover Active]" if f_a[idx] else ""
    print(f"  Patient {idx}: {val:.3f}{tag}")

## 5. Full 12-Finding Matrix Fusion Demo

Now let us run full matrix fusion across all 12 findings for 10 simulated studies:

In [ ]:
from rsna_knee import fuse_all

N_studies = 10
pred_a = rng.uniform(0.01, 0.99, size=(N_studies, 12))
pred_b = rng.uniform(0.01, 0.99, size=(N_studies, 12))
pred_c = rng.uniform(0.01, 0.99, size=(N_studies, 12))

# Study 3 failed in Model A -> filled with sentinel 0.5
pred_a[3, :] = 0.5

final_submission = fuse_all(pred_a, pred_b, pred_c, FINDINGS)
print(f"Submission Matrix Shape: {final_submission.shape}")
print(f"Value Range: [{final_submission.min():.3f}, {final_submission.max():.3f}]")
print(f"All values finite: {np.isfinite(final_submission).all()}")
print("\nFirst 2 studies across 12 findings:")
for i in range(2):
    scores_str = ", ".join(f"{f}: {final_submission[i, j]:.2f}" for j, f in enumerate(FINDINGS[:4]))
    print(f"  Study {i}: {scores_str} ...")